<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h1 style="background:linear-gradient(135deg,#22313d 0%,#284740 100%);color:#edf5f3;padding:18px 22px;border-radius:20px;border:1px solid #3a5255;border-left:10px solid #78b0a1;box-shadow:0 10px 24px rgba(0,0,0,0.20);margin:0 0 18px 0;">Comprehensive SFT Guide</h1>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">This notebook provides a comprehensive guide to Supervised Fine-Tuning (SFT) using LLaMA-Factory, covering:</p>
<ol style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>Dataset Types</strong>: Instruction-response, conversation, tool calling, multimodal</li>
<li style="margin:6px 0;"><strong>Training Methods</strong>: Full fine-tuning, LoRA, QLoRA</li>
<li style="margin:6px 0;"><strong>Advanced Features</strong>: Multi-turn conversations, custom datasets</li>
<li style="margin:6px 0;"><strong>Evaluation</strong>: Performance benchmarking and quality assessment</li>
</ol>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Table of Contents</h2>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#setup-and-installation">Setup and Installation</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#dataset-preparation">Dataset Preparation</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#training-configuration">Training Configuration</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#model-training">Model Training</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#evaluation-and-benchmarking">Evaluation and Benchmarking</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#advanced-features">Advanced Features</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#best-practices">Best Practices</a></li>
</ul>
</div>


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Setup and Installation</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">First, let's install the required dependencies and set up the environment.</p>
</div>


In [ ]:
# Install LLaMA-Factory and dependencies
%pip install -r requirements.txt
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Clone LLaMA-Factory (if not already cloned)
# !git clone https://github.com/hiyouga/LLaMA-Factory.git
# !cd LLaMA-Factory


In [ ]:
# Import required libraries
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from llamafactory import ChatModel
import json
import os

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Dataset Preparation</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">Let's prepare different types of datasets for SFT training.</p>
</div>


In [ ]:
# 1. Instruction-Response Dataset (Alpaca-style)
instruction_response_data = [
    {
        "instruction": "Explain the concept of machine learning in simple terms.",
        "input": "",
        "output": "Machine learning is a type of artificial intelligence that allows computers to learn and improve from experience without being explicitly programmed. It works by finding patterns in data and using those patterns to make predictions or decisions. Think of it like teaching a child - instead of giving step-by-step instructions, you show them many examples and they learn to recognize patterns and make decisions based on what they've seen."
    },
    {
        "instruction": "Write a Python function to calculate the factorial of a number.",
        "input": "n = 5",
        "output": "def factorial(n):\n    if n == 0 or n == 1:\n        return 1\n    else:\n        return n * factorial(n-1)\n\nresult = factorial(5)\nprint(result)  # Output: 120"
    }
]

# Save instruction-response dataset
with open('data/instruction_response_demo.json', 'w') as f:
    json.dump(instruction_response_data, f, indent=2)


In [ ]:
# 2. Conversation Dataset (ShareGPT-style)
conversation_data = [
    {
        "conversations": [
            {"from": "human", "value": "Hello! Can you help me understand quantum computing?"},
            {"from": "gpt", "value": "Hello! I'd be happy to help you understand quantum computing. Quantum computing is a revolutionary approach to computation that leverages the principles of quantum mechanics. Unlike classical computers that use bits (0s and 1s), quantum computers use quantum bits or 'qubits' that can exist in multiple states simultaneously thanks to a property called superposition.\n\nThis allows quantum computers to process vast amounts of information in parallel, making them potentially much faster for certain types of problems like:\n\n1. **Cryptography** - Breaking current encryption methods\n2. **Drug discovery** - Simulating molecular interactions\n3. **Optimization** - Finding optimal solutions to complex problems\n4. **Machine learning** - Training more sophisticated AI models\n\nWould you like me to explain any of these applications in more detail, or do you have questions about specific quantum concepts like superposition, entanglement, or quantum gates?"}
        ]
    }
]

# Save conversation dataset
with open('data/conversation_demo.json', 'w') as f:
    json.dump(conversation_data, f, indent=2)


In [ ]:
# 3. Tool Calling Dataset
tool_calling_data = [
    {
        "conversations": [
            {"from": "human", "value": "What's the weather like in New York today?"},
            {"from": "gpt", "value": "First, the user is asking about the weather in New York. I need to use the get_weather tool to fetch current weather information."}
        ],
        "tools": [
            {
                "name": "get_weather",
                "description": "Get current weather information for a city",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "city": {"type": "string", "description": "City name"}
                    },
                    "required": ["city"]
                }
            }
        ]
    }
]

# Save tool calling dataset
with open('data/tool_calling_demo.json', 'w') as f:
    json.dump(tool_calling_data, f, indent=2)
